# P5 Stage 1 — local run (ASUS A16, RTX 5050 8GB)

In [ ]:
!pip install -q datasets

import json
from datasets import load_dataset

print("Downloading SimpleQA dataset...")
ds = load_dataset("basicv8vc/SimpleQA", split="test")
sampled_ds = ds.shuffle(seed=42).select(range(200))

pool_v2 = []
for idx, row in enumerate(sampled_ds):
    pool_v2.append({"item_id": f"ITEM_{idx+1:03d}", "prompt": row["problem"], "gold_answer": row["answer"]})

with open("pool_v2.json", "w", encoding="utf-8") as f:
    json.dump(pool_v2, f, indent=2, ensure_ascii=False)
print(f"Created pool_v2.json with {len(pool_v2)} items.")

In [ ]:
# RTX 5050 is Blackwell (sm_120) -> needs the CUDA 12.8 PyTorch build.
# The default "pip install torch" will NOT use this GPU.
!pip install --index-url https://download.pytorch.org/whl/cu128 torch
!pip install -q transformers accelerate

# mamba-ssm / causal-conv1d are deliberately NOT installed.
# Mamba runs through transformers' built-in PyTorch path; fine for 400 short runs.

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print("If that is NOT your RTX 5050, you installed the wrong torch wheel.")
else:
    print("CUDA False -> wrong torch build. Reinstall with the cu128 index URL.")

In [ ]:
import os, json
assert os.path.exists("pool_v2.json"), "Run the first cell to create pool_v2.json."
pool_data = json.load(open("pool_v2.json", encoding="utf-8"))
print(f"Loaded pool: {len(pool_data)} items.")

In [ ]:
%%writefile p5_local_harness_v2.py
import argparse, json, os, random
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

FEWSHOT = (
    "Answer each question with a short, direct factual answer.\n\n"
    "Q: What is the capital of Japan?\nA: Tokyo.\n\n"
    "Q: Who wrote the novel Pride and Prejudice?\nA: Jane Austen.\n\n"
    "Q: What is the chemical symbol for gold?\nA: Au.\n\n"
)

def run_model(model_name, pool_path, output_path, mode="base"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading {model_name} onto {device}...")
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto")
    model.eval()
    items = json.load(open(pool_path, encoding="utf-8"))
    results = []
    if os.path.exists(output_path):
        try: results = json.load(open(output_path, encoding="utf-8"))
        except json.JSONDecodeError: results = []
    done = {r["item_id"] for r in results}
    print(f"Already completed: {len(done)} / {len(items)}")
    for idx, item in enumerate(items):
        if item["item_id"] in done: continue
        q = item.get("prompt", item.get("question", ""))
        inputs = tok(f"{FEWSHOT}Q: {q}\nA:", return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=64, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        resp = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        resp = resp.split("\nQ:")[0].strip()
        results.append({"item_id": item["item_id"], "prompt": q,
                        "gold_answer": item.get("gold_answer", item.get("answer", "")),
                        "response": resp, "model": model_name})
        json.dump(results, open(output_path, "w", encoding="utf-8"), indent=2)
        if (idx + 1) % 25 == 0 or (idx + 1) == len(items):
            print(f"Processed {idx + 1}/{len(items)}")
    print(f"Saved: {output_path}")

def make_blind_sheet(inputs, out_csv, key_csv):
    recs = []
    for fp in inputs: recs.extend(json.load(open(fp, encoding="utf-8")))
    random.seed(42); random.shuffle(recs)
    blind, key = [], []
    for i, r in enumerate(recs):
        rid = f"RESP_{i+1:04d}"
        blind.append({"response_id": rid, "prompt": r["prompt"],
                      "gold_answer": r["gold_answer"],
                      "model_response": r["response"], "failure_type": ""})
        key.append({"response_id": rid, "item_id": r["item_id"], "true_model": r["model"]})
    pd.DataFrame(blind).to_csv(out_csv, index=False)
    pd.DataFrame(key).to_csv(key_csv, index=False)
    print(f"Blind sheet: {out_csv} ({len(blind)} rows)")
    print(f"Sealed key : {key_csv}  -- do not open until kappa clears 0.80")
    print("failure_type codes: 0=correct 1=factual_error 2=abstention 3=degenerate")

if __name__ == "__main__":
    p = argparse.ArgumentParser(); sub = p.add_subparsers(dest="command")
    r = sub.add_parser("run"); r.add_argument("--model", required=True)
    r.add_argument("--mode", default="base"); r.add_argument("--pool", default="pool_v2.json")
    r.add_argument("--out", required=True)
    b = sub.add_parser("blind"); b.add_argument("--inputs", nargs="+", required=True)
    b.add_argument("--out", required=True); b.add_argument("--key", required=True)
    a = p.parse_args()
    if a.command == "run": run_model(a.model, a.pool, a.out, a.mode)
    elif a.command == "blind": make_blind_sheet(a.inputs, a.out, a.key)

In [ ]:
# ~5.6 GB each in fp16, one at a time -> fits 8 GB.
!python p5_local_harness_v2.py run --model EleutherAI/pythia-2.8b     --mode base --out pythia_2_8b.json
!python p5_local_harness_v2.py run --model state-spaces/mamba-2.8b-hf --mode base --out mamba_2_8b.json

In [ ]:
!python p5_local_harness_v2.py blind \
  --inputs pythia_2_8b.json mamba_2_8b.json \
  --out p5_scoresheet_BLIND.csv \
  --key p5_key_SEALED.csv

In [ ]:
%%writefile cohens_kappa.py
import sys
import pandas as pd
from sklearn.metrics import cohen_kappa_score
if len(sys.argv) < 3:
    print("Usage: python cohens_kappa.py SADI_scored.csv DEWAN_scored.csv"); sys.exit(1)
df1 = pd.read_csv(sys.argv[1]).sort_values("response_id")
df2 = pd.read_csv(sys.argv[2]).sort_values("response_id")
assert len(df1) == len(df2), "Row counts do not match!"
assert (df1["response_id"].values == df2["response_id"].values).all(), "response_ids mismatch!"
k = cohen_kappa_score(df1["failure_type"].astype(int), df2["failure_type"].astype(int))
print("="*40); print(f"Cohen's Kappa: {k:.4f}"); print("="*40)
print("PASSED (>=0.80). Ready to unblind." if k >= 0.80 else "FAILED (<0.80). Reconcile and re-score.")